# MiniCode Parte 2 — QLoRA para seleção de ferramentas
Execute em Google Colab com GPU. Este notebook não afirma que treinamento foi executado; ele cria artefatos somente quando suas células forem executadas.

In [1]:
!pip -q install -U transformers datasets peft accelerate trl bitsandbytes matplotlib
from google.colab import drive
drive.mount('/content/drive')
# Copie/clonar o repositório e ajuste este caminho sem incluir credenciais.
REPO = '/content/MiniCode'
OUT = '/content/drive/MyDrive/minicode-tool-selector'


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [7]:
import json, sys
from pathlib import Path
sys.path.insert(0, str(Path(REPO) / 'src'))
from minicode.specialization_data import load_examples, validate_examples
rows = load_examples(Path(REPO) / 'data/tool_selection/examples.jsonl')
print(validate_examples(rows))
TOOLS = ['list_files','read_file','write_file','replace_text','replace_python_function','search_code','run_tests','read_test_output']
def prompt(row, include_context=True):
    x = dict(row['input'])
    if not include_context: x['specification'], x['plan'] = {}, {}
    return 'Escolha a próxima ferramenta. Responda apenas o nome.\n' + json.dumps(x, ensure_ascii=False) + '\nFerramentas: ' + ', '.join(TOOLS)
splits = {s:[r for r in rows if r['split']==s] for s in ('train','validation','test')}
assert not {r['task_family'] for r in splits['train']} & {r['task_family'] for r in splits['test']}


ModuleNotFoundError: No module named 'minicode'

In [ ]:
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, TrainingArguments
from peft import LoraConfig
from trl import SFTTrainer
MODEL = 'Qwen/Qwen2.5-0.5B-Instruct'
tokenizer = AutoTokenizer.from_pretrained(MODEL, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
def as_dataset(examples):
    return Dataset.from_list([{'text': tokenizer.apply_chat_template([{'role':'user','content':prompt(r)}, {'role':'assistant','content':r['label']}], tokenize=False)} for r in examples])
quant = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4', bnb_4bit_compute_dtype='float16')
model = AutoModelForCausalLM.from_pretrained(MODEL, quantization_config=quant, device_map='auto', trust_remote_code=True)
lora = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, target_modules=['q_proj','k_proj','v_proj','o_proj'], task_type='CAUSAL_LM')


In [ ]:
args = TrainingArguments(output_dir=OUT, num_train_epochs=5, learning_rate=2e-4, per_device_train_batch_size=1, gradient_accumulation_steps=8, eval_strategy='epoch', logging_steps=1, save_strategy='epoch', report_to='none', fp16=True)
trainer = SFTTrainer(model=model, args=args, train_dataset=as_dataset(splits['train']), eval_dataset=as_dataset(splits['validation']), processing_class=tokenizer, peft_config=lora, dataset_text_field='text', max_seq_length=1024)
trainer.train()
trainer.save_model(OUT + '/adapter')
trainer.save_state()
# Curvas reais, após train():
import matplotlib.pyplot as plt
hist = trainer.state.log_history
plt.plot([x['step'] for x in hist if 'loss' in x], [x['loss'] for x in hist if 'loss' in x], label='train')
plt.plot([x['step'] for x in hist if 'eval_loss' in x], [x['eval_loss'] for x in hist if 'eval_loss' in x], label='validation')
plt.legend(); plt.xlabel('step'); plt.ylabel('loss'); plt.show()


In [ ]:
# Rode duas vezes, com MODEL base e com adapter carregado, sem olhar o split test para ajustar hiperparâmetros.
def predict(model, examples):
    out=[]
    for r in examples:
        inputs=tokenizer(prompt(r), return_tensors='pt').to(model.device)
        answer=tokenizer.decode(model.generate(**inputs, max_new_tokens=16, do_sample=False)[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True).strip().split()[0]
        out.append({'id':r['id'], 'tool':answer})
    return out
# Path(OUT + '/lora_test_predictions.jsonl').write_text('\n'.join(json.dumps(x) for x in predict(model, splits['test']))+'\n')
# !python {REPO}/scripts/evaluate_tool_selection.py --predictions {OUT}/lora_test_predictions.jsonl --output {OUT}/lora_test_report.json
